# Does CSRT nose tracking improve our respiration model?
**Executed experiment · 17 October 2026**

We borrow [JoulesEye's](https://sustainability-lab.github.io/jouleseye/) tracking idea,
using [OpenCV CSRT](https://docs.opencv.org/4.6.0/d2/da2/classcv_1_1TrackerCSRT.html)
with the existing manually inspected first-frame boxes. RGB initialization is
unavailable in this dataset. This notebook presents actual saved experiment
outputs; reading those reports is executed below. Synthetic test outputs are
kept separate from real-data results. The source listings and opt-in rerun cells
make the extraction/training steps reviewable and reproducible.

The original fixed-crop walkthrough is in `thermal_respiration_walkthrough.ipynb`.
This local report embeds participant thermal frames.

# Fixed versus tracked nasal crops: experiment protocol

Specified before inspecting respiration errors for the tracked pipeline, 17 October 2026.

## Change being tested

Borrow JoulesEye's use of CSRT to follow the nasal region. Initialize from the
existing manually specified first-frame box in `config/nose_boxes.json` because
the supplied recordings contain no synchronized RGB. Use OpenCV 4.12.0 CSRT
defaults, including scale adaptation. Do not optimize tracker settings, crop
sizes, missing-data thresholds, or neural hyperparameters using test errors.
This is a thermal-only adaptation, not an exact JoulesEye reproduction.

Read original 8-bit frames and original CSV time. At each frame, compute the same
nine patch means from both the fixed and tracked box. Preserve every timestamp.
Skip CSRT updates on identical consecutive frames; retain the previous tracking
status and apply the existing >=5-frame frozen-run exclusion afterward. Mark
failed, nonfinite, undersized, or out-of-image boxes missing. Never substitute
the fixed crop after failure. Continue CSRT updates to allow recovery, without
claiming detector-based reinitialization. A successful tracker update is not
proof that the box still covers the nose.

Recompute fixed traces and require exact numerical equality to the earlier
`artifacts/traces` arrays. Write new outputs separately under
`artifacts/tracking_experiment/{fixed,tracked}`. Record per-frame boxes and
status, trace hashes, raw pixel hashes, source hashes, failures and runtime.

## Evaluation

- Keep the four previously eligible people; exclude `anestis` for unresolved alignment.
- Keep 20-second windows, five-second stride, 10 Hz resampling, marker-1 labels,
  normalization, missing threshold and 60-second condition boundary unchanged.
- Use the intersection of fixed/tracked eligible windows for **both training
  and testing**, separately for each person. Report each variant's original
  candidate coverage and exclusions; do not silently omit a person if all their
  paired windows are unavailable.
- Train both CNN and CNN + GRU from scratch for 40 epochs per held-out-person
  fold. Use the existing optimizer/loss/batch settings unchanged.
- Primary comparison: CNN + GRU, seed 7, macro MAE across held-out people.
- Predetermined sensitivity runs: seeds 17 and 27, for both architectures and crops.
- Include the training-median and spectral comparators. Score each method on
  matching windows with predictions from both crops, so spectral abstention
  cannot change the neural methods' scoring subset.
- Report overall, per-person, and still/moving/transition errors; per-seed paired
  changes; mean and population standard deviation over seeds. Negative
  tracked-minus-fixed MAE means improvement. Repeated seeds are not independent
  participants and do not provide population-level confidence intervals.
- Do not compare a coverage-reduced result directly with the old all-window
  2.986 BPM figure. Compare the retrained paired fixed baseline instead.

## References

- [JoulesEye project and method](https://sustainability-lab.github.io/jouleseye/)
- [OpenCV CSRT API](https://docs.opencv.org/4.6.0/d2/da2/classcv_1_1TrackerCSRT.html)

The main research question is whether this concrete adaptation improves this
pilot's measured error. It cannot establish reliable generalization from four
people or validate the anatomical accuracy of tracking without frame labels.

In [ ]:
from pathlib import Path
import json, sys, hashlib
import numpy as np
from IPython.display import display, Markdown, Image
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'thermal_face').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
FOLDER = ROOT / 'artifacts' / 'tracking_experiment'
report = json.loads((FOLDER / 'comparison.json').read_text())
audit = json.loads((FOLDER / 'tracking_audit.json').read_text())
print('Recorded training environment:', json.dumps(report['environment'], indent=2))
print('Tracker:', audit['tracker'], '| OpenCV:', audit['opencv'])
print('Seeds:', report['protocol']['seeds'], '| epochs:', report['protocol']['epochs'])
print('Training/evaluation wall seconds:', round(report['elapsed_seconds'], 2))
print('Extraction wall seconds:', round(sum(s['elapsed_seconds'] for s in audit['subjects'].values()), 2))
# Verify that the reports refer to the trace archives currently present.
for variant, hashes in report['trace_sha256'].items():
    for person, expected in hashes.items():
        assert hashlib.sha256((FOLDER / variant / f'{person}.npz').read_bytes()).hexdigest() == expected
print('All trace hashes verified.')

## Reproduce extraction and training (opt-in)
The saved results shown below were generated using these module entry points.
`RERUN=False` keeps a presentation from launching several minutes of tracking.
For this workspace only, OpenCV contrib was installed in
`artifacts/tracking_dependencies`; a clean environment can instead install
`requirements-tracking.txt`. Restart the notebook kernel after changing OpenCV
packages if an older cv2 module has already been imported.

In [ ]:
RERUN = False
if RERUN:
    local_cv = ROOT / 'artifacts' / 'tracking_dependencies'
    if local_cv.is_dir():
        sys.path.insert(0, str(local_cv))
    from thermal_face.tracking import extract
    from thermal_face.compare_tracking import run
    audit = extract(ROOT / 'Manuel_clean', FOLDER, ROOT / 'config' / 'nose_boxes.json',
                    ROOT / 'artifacts' / 'traces')
    report = run(FOLDER, FOLDER / 'comparison.json', seeds=[7, 17, 27], epochs=40, device='cpu')
else:
    print('Displaying the completed extraction and training reports; no retraining during this export.')

## Tracking behavior and missing-data accounting
Frozen runs keep their original time coordinates. Failed boxes would become
missing samples; they are never replaced by a fixed crop. Duplicate frames do
not update the tracker. The nine patch features, timestamp resampling and
window rules remain the same. CSRT includes scale adaptation, so patch areas
can change with the box dimensions. A zero failure count does not exclude drift.

In [ ]:
table = '| Person | Frames | Frozen | Failed tracking | Valid tracked | Fixed matches prior |\n|---|---:|---:|---:|---:|---|\n'
for person, s in audit['subjects'].items():
    table += f"| {person} | {s['frames']} | {s['frozen_frames']} | {s['tracking_failed_frames']} | {s['valid_tracked_frames']} | {s['fixed_matches_previous_trace_exactly']} |\n"
display(Markdown(table))
for variant, people in report['coverage'].items():
    print(variant, {p: {k: s[k] for k in ['candidates', 'eligible', 'paired']} for p, s in people.items()})

## Visual inspection: fixed box versus tracked box
Red = fixed starting box; green = CSRT box. Samples are selected at predetermined
elapsed times (0, 30, 60, 90, 115 seconds), not selected based on prediction error.
This qualitative inspection cannot replace annotated landmark accuracy.
Joao's 90 s and 115 s boxes visibly drift to the lower face/neck, while
Claudio's move toward the upper face/eye area. These updates are not flagged as
failures by CSRT. No settings were adjusted in response to these observations.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image as PILImage
import io
def show_plot(fig):
    stream = io.BytesIO()
    fig.savefig(stream, format='png', dpi=140, bbox_inches='tight')
    plt.close(fig)
    display(Image(data=stream.getvalue()))
boxes = json.loads((ROOT / 'config' / 'nose_boxes.json').read_text())
people = sorted(audit['subjects'])
fig, axes = plt.subplots(len(people), 5, figsize=(14, 10))
for row, person in enumerate(people):
    paths = sorted((ROOT / 'Manuel_clean' / person).glob('frame_*.png'))
    with np.load(FOLDER / 'tracked' / f'{person}.npz') as z:
        seconds, tracked_boxes, valid = z['seconds'], z['boxes'], z['track_ok']
    for col, time_s in enumerate([0, 30, 60, 90, 115]):
        i = int(np.argmin(abs(seconds-time_s)))
        with PILImage.open(paths[i]) as picture:
            axes[row, col].imshow(np.array(picture), cmap='gray')
        for box, color in [(boxes[person], 'red'), (tracked_boxes[i], 'lime')]:
            if box[0] >= 0:
                x, y, w, h = box
                axes[row, col].add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=1.3))
        axes[row, col].set_title(f'{person} {seconds[i]:.1f}s', fontsize=9)
        axes[row, col].axis('off')
fig.suptitle('Fixed (red) and tracked (green) nasal crops; no label-based selection')
fig.tight_layout()
show_plot(fig)

## CSRT tracking experiment — measured results

Executed on the restored original recordings, 17 October 2026. This is a
thermal-only adaptation of JoulesEye's tracking idea: the existing first-frame
boxes initialize CSRT; there is no RGB detector or detector-based recovery.
The complete [executed experiment notebook](tracking_experiment.ipynb) and
[HTML presentation](tracking_experiment.html) include code, per-frame diagnostics,
sample overlays, paired evaluation and limitations.

**Primary result (CNN + GRU, seed 7):** fixed crops **2.986 BPM** macro MAE;
tracked crops **2.969 BPM**. Tracking reduced
error by **0.017 BPM** (0.6%).

| Method | Fixed MAE | Tracked MAE | Tracked − fixed |
|---|---:|---:|---:|
| train_median | 2.501 | 2.501 | +0.000 |
| spectral | 10.618 | 10.809 | +0.190 |
| cnn | 3.577 | 3.268 | -0.309 |
| cnn_gru | 2.986 | 2.969 | -0.017 |

### Predetermined seed sensitivity

| Seed | Fixed CNN + GRU | Tracked CNN + GRU | Tracked − fixed |
|---|---:|---:|---:|
| 7 | 2.986 | 2.969 | -0.017 |
| 17 | 14.867 | 14.865 | -0.002 |
| 27 | 2.557 | 2.551 | -0.006 |


Across the three predetermined seeds (7, 17, 27), CNN + GRU mean MAE is
**6.803 BPM fixed** versus
**6.795 BPM tracked**.
Tracking improves 3 of 3 seeds.
These seeds measure optimization sensitivity, not additional independent people.

The primary CNN + GRU gain is very small. Seed 17 produces much larger errors
for both crop variants than seeds 7 and 27; this indicates substantial training
sensitivity under the unchanged 40-epoch recipe. The three-seed mean must not be
presented without that context. Tracking does not solve this optimization issue.

For the CNN-only comparison, three-seed mean MAE changes from
3.682 to
3.247 BPM, with improvement in
3/3 seeds. This is a larger
numerical benefit, but the tracked CNN still does not beat the median baseline
on the three-seed mean. The spectral comparator becomes worse with tracking.

### Coverage and fairness

- Fixed eligible windows: 77/84; tracked: 77/84.
- Both training and testing use the same 77 paired eligible windows,
  with leave-one-person-out splits and unchanged architectures, loss, optimizer,
  epoch count, normalization and reference labels.
- All four recomputed fixed traces matched the prior trace arrays exactly.
- `anestis` remains excluded because its frame/reference alignment is unresolved.
- Explicit CSRT failures: 0 frames.
  Successful updates are not a measurement of anatomical tracking accuracy.

### Visual finding: undetected drift

Inspection of predetermined 90 s and 115 s samples shows Joao's tracked box
moving onto the lower face/neck rather than remaining on the nose. Claudio's
90 s and 115 s samples shift toward the upper face/eye area. CSRT still returns
successful updates on these frames. Thus a lower respiration error, if present,
does not by itself validate nasal tracking or prove that the learned signal is
physiological breathing. The notebook includes these overlays for review.

### Interpretation and limitations

This experiment supports a reduction
in the primary model's seed-7 error on this pilot. It is not evidence that tracking
will generalize to new recordings. Only four people, one recording each, were
evaluated. Fixed initial boxes, possible undetected drift, uncertain marker
semantics, and the assumed motion boundary remain limitations. CSRT changes both
position and scale, so this experiment evaluates that combined change.

The training-median comparator remains essential: its paired seed-7 MAE is
2.501 BPM;
the tracked CNN + GRU does not beat it.
No tracker or model settings were revised after inspecting these errors.

Next, annotate nose locations through motion to quantify drift, test localization
and explicit recovery, verify reference events/alignment, and collect additional
people and sessions. Do not replace a baseline based solely on a favorable seed.

### Reproduction

Install `requirements-tracking.txt` in a clean environment (use the contrib
OpenCV distribution rather than combining multiple OpenCV wheels), then run:

```powershell
python -m thermal_face.tracking --root Manuel_clean --boxes config/nose_boxes.json --out artifacts/tracking_experiment --fixed-reference artifacts/traces
python -m thermal_face.compare_tracking --traces artifacts/tracking_experiment --epochs 40 --seeds 7 17 27 --device cpu
python docs/build_tracking_report.py
```

Machine-readable reports: `artifacts/tracking_experiment/tracking_audit.json`
and `artifacts/tracking_experiment/comparison.json`. The protocol is documented
in [tracking_experiment_protocol.md](tracking_experiment_protocol.md).

## Per-person and condition results, primary seed 7
All errors are breaths/minute. Each person contributes equally to macro MAE.
The four people are the independent evaluation units; overlapping windows and
repeated seeds do not enlarge that number.

In [ ]:
primary = report['runs'][0]
table = '| Person | Fixed CNN + GRU | Tracked CNN + GRU | Change |\n|---|---:|---:|---:|\n'
for person in sorted(primary['comparison']['cnn_gru']['fixed']['per_subject_mae_bpm']):
    a, b = [primary['comparison']['cnn_gru'][v]['per_subject_mae_bpm'][person] for v in ['fixed', 'tracked']]
    table += f'| {person} | {a:.3f} | {b:.3f} | {b-a:+.3f} |\n'
display(Markdown(table))
table = '| Condition | Fixed CNN + GRU | Tracked CNN + GRU |\n|---|---:|---:|\n'
for condition in ['still', 'moving', 'transition']:
    vals = [primary['variants'][v]['by_condition'][condition]['methods']['cnn_gru']['macro_mae_bpm'] for v in ['fixed', 'tracked']]
    table += '| ' + condition + ' | ' + ' | '.join(f'{v:.3f}' if v is not None else 'N/A' for v in vals) + ' |\n'
display(Markdown(table))
fig, ax = plt.subplots(figsize=(9, 4))
methods = list(primary['comparison'])
positions = np.arange(len(methods))
for offset, variant, color in [(-0.18, 'fixed', '#637c91'), (0.18, 'tracked', '#188c7b')]:
    vals = [primary['comparison'][m][variant]['macro_mae_bpm'] for m in methods]
    bars = ax.bar(positions+offset, vals, 0.36, label=variant, color=color)
    ax.bar_label(bars, fmt='%.2f', padding=3)
ax.set(xticks=positions, xticklabels=methods, ylabel='Macro MAE (BPM)', title='Measured paired comparison: seed 7')
ax.legend()
show_plot(fig)

## Verify implementation checks
This output is the saved test log from the implementation run. It includes
synthetic translated-target tracking, failure/recovery handling, frozen-run
timing, matched train/test windows, and existing evaluation/model tests.
These checks do not establish anatomical accuracy or clinical validity.

In [ ]:
text = (ROOT / 'artifacts' / 'tracking_tests.txt').read_text(encoding='utf-8')
lines = text.splitlines()
print('\n'.join(line for line in lines if line.startswith('test_')))
print('\n'.join(lines[-5:]))

## Implementation source used for this experiment
Reference listings; rerun cells above call these modules.

### `thermal_face/tracking.py`

```python
"""CSRT nasal tracking experiment, initialized from the existing first-frame box.

No RGB initialization, detector, label-based tuning, or manual recovery is used.
Unsuccessful/out-of-frame boxes are missing measurements, never fixed-box fallback.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
from time import perf_counter

import numpy as np

from .data import SUBJECTS, read_respiration


def create_tracker():
    import cv2
    factory = getattr(cv2, 'TrackerCSRT_create', None)
    if factory is None:
        factory = getattr(getattr(cv2, 'legacy', None), 'TrackerCSRT_create', None)
    if factory is None:
        raise RuntimeError('CSRT requires opencv-contrib-python; the base opencv-python package is insufficient')
    return factory()  # Published OpenCV defaults; no tuning against respiration labels.


def integer_box(box, shape):
    """Reject invalid boxes rather than silently changing the region by clipping."""
    values = np.asarray(box, dtype=float)
    if values.shape != (4,) or not np.isfinite(values).all():
        return None
    x, y, w, h = (int(round(v)) for v in values)
    if min(w, h) < 3 or min(x, y) < 0 or x + w > shape[1] or y + h > shape[0]:
        return None
    return x, y, w, h


def patch_means(frame, box):
    x, y, w, h = box
    crop = frame[y:y+h, x:x+w].astype(np.float32)
    return np.asarray([crop[r*h//3:(r+1)*h//3, c*w//3:(c+1)*w//3].mean()
                       for r in range(3) for c in range(3)], dtype=np.float32)


def unfrozen_mask(repeated):
    """Match the fixed extractor's >=5 identical-frame exclusion exactly."""
    valid = np.ones(len(repeated), dtype=bool)
    start = None
    for i in range(1, len(repeated) + 1):
        same = i < len(repeated) and repeated[i]
        if same and start is None:
            start = i - 1
        elif not same and start is not None:
            if i - start >= 5:
                valid[start:i] = False
            start = None
    return valid


class NoseTracker:
    """Track only images; no access to subject labels or reference events."""
    def __init__(self, frame, box, factory=create_tracker):
        self.tracker = factory()
        self.box = integer_box(box, frame.shape)
        if self.box is None:
            raise ValueError('Initial box must fit inside the image and contain nine nonempty patches')
        result = self.tracker.init(frame, self.box)
        if result is False:
            raise RuntimeError('CSRT initialization failed')
        self.ok = True

    def update(self, frame, repeated=False):
        if repeated:
            return self.box if self.ok else None, 'duplicate' if self.ok else 'lost'
        was_ok = self.ok
        success, candidate = self.tracker.update(frame)
        box = integer_box(candidate, frame.shape) if success else None
        self.ok = box is not None
        if self.ok:
            self.box = box
        # Continue asking CSRT to update after loss; no synthetic/reinitialized boxes.
        return box, ('tracked' if was_ok else 'recovered') if self.ok else 'lost'


def extract(root: Path, out: Path, boxes_path: Path, fixed_reference: Path | None = None,
            subjects=SUBJECTS):
    import cv2
    cv2.setNumThreads(2)
    cv2.setRNGSeed(7)
    boxes = json.loads(boxes_path.read_text(encoding='utf-8'))
    out.mkdir(parents=True, exist_ok=True)
    for variant in ['fixed', 'tracked']:
        (out / variant).mkdir(exist_ok=True)
    report = {
        'tracker': 'OpenCV CSRT, default parameters', 'opencv': cv2.__version__,
        'initialization': 'existing hand-inspected first-frame box; no RGB',
        'failure_policy': 'missing; no fixed-box fallback; subsequent CSRT updates may recover',
        'freeze_policy': 'skip tracker update on identical frames; flag runs >=5 as missing',
        'box_policy': 'CSRT position and scale; rounded integer boxes; reject outside image',
        'seed': 7, 'opencv_threads': 2, 'subjects': {}, 'excluded': {},
        'boxes_sha256': hashlib.sha256(boxes_path.read_bytes()).hexdigest(),
        'source_sha256': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
    }
    for subject in subjects:
        paths = sorted((root / subject).glob('frame_*.png'))
        times, markers = read_respiration(next(root.glob(f'*{subject}_respiration.csv')))
        if len(paths) != len(times):
            report['excluded'][subject] = f'{len(paths)} frames versus {len(times)} reference rows'
            print(f'Skipping {subject}: unverified alignment', flush=True)
            continue
        if not paths:
            raise ValueError(f'No frames for {subject}')
        started = perf_counter()
        fixed = np.zeros((len(paths), 9), dtype=np.float32)
        tracked = np.zeros_like(fixed)
        repeated = np.zeros(len(paths), dtype=bool)
        track_ok = np.zeros(len(paths), dtype=bool)
        track_boxes = np.full((len(paths), 4), -1, dtype=np.int32)
        states = np.empty(len(paths), dtype='U10')
        previous = None
        pixels_hash = hashlib.sha256()
        print(f'{subject}: extracting and tracking {len(paths)} original frames', flush=True)
        for i, path in enumerate(paths):
            frame = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
            if frame is None:
                raise ValueError(f'Unreadable image: {path}')
            if frame.ndim == 3 and frame.shape[2] == 3:
                if not (np.array_equal(frame[:, :, 0], frame[:, :, 1]) and
                        np.array_equal(frame[:, :, 0], frame[:, :, 2])):
                    raise ValueError(f'Expected identical grayscale channels: {path}')
                gray = frame[:, :, 0]
            elif frame.ndim == 2:
                gray = frame
                frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
            else:
                raise ValueError(f'Unsupported frame: {path}')
            if gray.dtype != np.uint8:
                raise ValueError('This experiment expects the original 8-bit images')
            pixels_hash.update(gray.tobytes())
            fixed_box = integer_box(boxes[subject], gray.shape)
            if fixed_box is None:
                raise ValueError(f'Invalid configured box for {subject}')
            fixed[i] = patch_means(gray, fixed_box)
            if i == 0:
                tracker = NoseTracker(frame, fixed_box)
                box, state = fixed_box, 'initial'
            else:
                repeated[i] = np.array_equal(gray, previous)
                box, state = tracker.update(frame, bool(repeated[i]))
            previous = gray.copy()
            states[i] = state
            if box is not None:
                track_ok[i] = True
                track_boxes[i] = box
                tracked[i] = patch_means(gray, box)
            if (i + 1) % 500 == 0:
                print(f'  {subject}: {i+1}/{len(paths)} frames; {np.count_nonzero(~track_ok[:i+1])} failed boxes; '
                      f'{perf_counter()-started:.1f}s', flush=True)
        valid = unfrozen_mask(repeated)
        seconds = np.array([(t-times[0]).total_seconds() for t in times])
        base = dict(seconds=seconds, marker=np.asarray(markers, dtype=np.int8))
        comparison = None
        if fixed_reference is not None:
            reference = fixed_reference / f'{subject}.npz'
            with np.load(reference, allow_pickle=False) as saved:
                comparison = all(np.array_equal(saved[k], v) for k, v in
                                 (base | {'trace': fixed, 'valid': valid}).items())
            if not comparison:
                raise AssertionError(f'Fixed extraction differs from the existing baseline: {subject}')
        np.savez_compressed(out / 'fixed' / f'{subject}.npz', **base, trace=fixed, valid=valid)
        np.savez_compressed(out / 'tracked' / f'{subject}.npz', **base,
                            trace=tracked, valid=valid & track_ok,
                            track_ok=track_ok, boxes=track_boxes, state=states, frozen=~valid)
        info = {
            'frames': len(paths), 'frozen_frames': int((~valid).sum()),
            'tracking_failed_frames': int((~track_ok).sum()),
            'tracking_success_fraction': float(track_ok.mean()),
            'valid_tracked_frames': int((valid & track_ok).sum()),
            'recoveries': int((states == 'recovered').sum()),
            'source_pixel_sha256': pixels_hash.hexdigest(),
            'reference_csv_sha256': hashlib.sha256(next(root.glob(f'*{subject}_respiration.csv')).read_bytes()).hexdigest(),
            'fixed_matches_previous_trace_exactly': comparison,
            'elapsed_seconds': perf_counter()-started,
        }
        report['subjects'][subject] = info
        (out / 'tracking_audit.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
        print(f'{subject} completed: {info}', flush=True)
    (out / 'tracking_audit.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--root', type=Path, required=True)
    parser.add_argument('--out', type=Path, default=Path('artifacts/tracking_experiment'))
    parser.add_argument('--boxes', type=Path, default=Path('config/nose_boxes.json'))
    parser.add_argument('--fixed-reference', type=Path)
    args = parser.parse_args()
    extract(args.root, args.out, args.boxes, args.fixed_reference)


if __name__ == '__main__':
    main()

```

### `thermal_face/compare_tracking.py`

```python
"""Paired fixed-versus-CSRT evaluation on identical training and test windows."""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
from time import perf_counter

import numpy as np

from .baselines import spectral_rate
from .evaluate import METHODS, summarize
from .train_rate import execution_environment, fit_predict
from .windows import load_window_records


def paired_windows(fixed, tracked):
    """Use the eligibility intersection for BOTH training and testing."""
    if set(fixed) != set(tracked) or len(fixed) < 3:
        raise ValueError('Need the same three or more subjects in both variants')
    paired = {}
    for subject in sorted(fixed):
        a = {w.start_s: w for w in fixed[subject]}
        b = {w.start_s: w for w in tracked[subject]}
        if a.keys() != b.keys():
            raise ValueError(f'Candidate timestamps differ for {subject}')
        for start in a:
            wa, wb = a[start], b[start]
            if (wa.end_s, wa.condition, wa.reference_bpm, wa.reference_rejection) != (
                    wb.end_s, wb.condition, wb.reference_bpm, wb.reference_rejection):
                raise ValueError(f'Reference labels or conditions differ for {subject} at {start}')
        paired[subject] = [start for start in sorted(a) if a[start].eligible and b[start].eligible]
        if not paired[subject]:
            raise ValueError(f'No paired eligible windows for {subject}; cannot silently drop a person')
    return paired


def run(folder: Path, out: Path, seeds=(7, 17, 27), epochs=40, device='cpu'):
    seeds = tuple(seeds)
    if not seeds or len(set(seeds)) != len(seeds) or epochs < 1:
        raise ValueError('Specify distinct seeds and positive epochs')
    started = perf_counter()
    records = {variant: load_window_records(folder / variant) for variant in ['fixed', 'tracked']}
    paired = paired_windows(records['fixed'], records['tracked'])
    people = sorted(paired)
    coverage = {}
    for variant in records:
        coverage[variant] = {}
        for person, windows in records[variant].items():
            coverage[variant][person] = {
                'candidates': len(windows), 'eligible': sum(w.eligible for w in windows),
                'paired': len(paired[person]),
                'windows': [w.metadata() for w in windows],
            }
    report = {
        'protocol': {
            'primary_seed': seeds[0], 'seeds': list(seeds), 'epochs': epochs,
            'split': 'leave-one-person-out',
            'paired_training_and_test': True,
            'paired_rule': 'intersection of fixed/tracked eligibility by subject and exact window timestamp',
            'window_s': 20, 'step_s': 5, 'sample_hz': 10,
            'model_selection': 'no selection or tuning using held-out results',
            'interpretation': 'seed variation is not independent-person uncertainty; four-person exploratory pilot',
            'excluded_subjects': {'anestis': 'unresolved frame/reference alignment'},
        },
        'environment': execution_environment(device), 'coverage': coverage,
        'paired_starts': paired, 'runs': [],
        'trace_sha256': {variant: {person: hashlib.sha256((folder / variant / f'{person}.npz').read_bytes()).hexdigest()
                                    for person in people} for variant in records},
        'source_sha256': {name: hashlib.sha256(Path(__file__).with_name(name).read_bytes()).hexdigest()
                          for name in ['compare_tracking.py', 'train_rate.py', 'models.py', 'windows.py', 'baselines.py']},
    }
    for seed in seeds:
        seed_result = {'seed': seed, 'variants': {}}
        for variant in records:
            examples = {person: [w for w in records[variant][person] if w.start_s in paired[person]] for person in people}
            rows, folds = [], []
            for held_out in people:
                training_people = [p for p in people if p != held_out]
                train = [(w.x, np.float32(w.reference_bpm)) for p in training_people for w in examples[p]]
                test = examples[held_out]
                test_x = np.stack([w.x for w in test])
                median = float(np.median(np.array([y for _, y in train], dtype=np.float32)))
                print(f'seed={seed} {variant} held_out={held_out}: train={len(train)}, test={len(test)}', flush=True)
                predicted = {}
                parameters = {}
                for architecture in ['cnn', 'cnn-gru']:
                    values, count = fit_predict(train, test_x, epochs, seed, architecture, device)
                    if np.shape(values) != (len(test),) or not np.isfinite(values).all():
                        raise ValueError('Nonfinite or malformed neural predictions; do not silently exclude them')
                    predicted[architecture.replace('-', '_')] = values
                    parameters[architecture] = count
                for i, window in enumerate(test):
                    spectrum, reason = spectral_rate(window.x)
                    row = window.metadata() | {
                        'predictions_bpm': {'train_median': median, 'spectral': spectrum,
                                            'cnn': float(predicted['cnn'][i]),
                                            'cnn_gru': float(predicted['cnn_gru'][i])},
                        'abstention_reasons': dict.fromkeys(METHODS),
                    }
                    row['abstention_reasons']['spectral'] = reason
                    rows.append(row)
                folds.append({'held_out': held_out, 'train_subjects': training_people,
                              'train_windows': len(train), 'test_windows': len(test),
                              'train_median_bpm': median, 'parameters': parameters})
            seed_result['variants'][variant] = {
                'folds': folds, 'windows': rows, 'overall': summarize(rows),
                'by_condition': {c: summarize([r for r in rows if r['condition'] == c]) for c in ['still', 'moving', 'transition']},
                'by_subject': {p: summarize([r for r in rows if r['subject'] == p]) for p in people},
            }
        # Pair predictions separately per method, so spectral abstention cannot affect CNN scoring.
        comparisons = {}
        variants = seed_result['variants']
        index = {v: {(r['subject'], r['start_s']): r for r in variants[v]['windows']} for v in variants}
        for method in METHODS:
            keys = [k for k in index['fixed'] if all(index[v][k]['predictions_bpm'][method] is not None for v in index)]
            from .evaluate import error_metrics
            metrics = {v: error_metrics([index[v][k] for k in keys], method) for v in index}
            a, b = [metrics[v]['macro_mae_bpm'] for v in ['fixed', 'tracked']]
            comparisons[method] = {
                **metrics, 'paired_prediction_windows': len(keys),
                'tracked_minus_fixed_mae_bpm': b-a if a is not None and b is not None else None,
            }
        seed_result['comparison'] = comparisons
        report['runs'].append(seed_result)
        print('Seed comparison:', json.dumps(comparisons, indent=2), flush=True)
    report['aggregate'] = {}
    for method in METHODS:
        values = np.array([[r['comparison'][method][v]['macro_mae_bpm'] for v in ['fixed', 'tracked']]
                           for r in report['runs']], dtype=float)
        if not np.isfinite(values).all():
            report['aggregate'][method] = {'status': 'no paired predictions in at least one seed'}
            continue
        differences = values[:, 1] - values[:, 0]
        report['aggregate'][method] = {
            'fixed_mean_mae_bpm': float(values[:, 0].mean()),
            'tracked_mean_mae_bpm': float(values[:, 1].mean()),
            'fixed_seed_sd_bpm': float(values[:, 0].std()),
            'tracked_seed_sd_bpm': float(values[:, 1].std()),
            'mean_tracked_minus_fixed_bpm': float(differences.mean()),
            'seeds_with_improvement': int((differences < 0).sum()),
        }
    report['elapsed_seconds'] = perf_counter()-started
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--traces', type=Path, default=Path('artifacts/tracking_experiment'))
    parser.add_argument('--out', type=Path, default=Path('artifacts/tracking_experiment/comparison.json'))
    parser.add_argument('--epochs', type=int, default=40)
    parser.add_argument('--seeds', type=int, nargs='+', default=[7, 17, 27])
    parser.add_argument('--device', choices=['cpu', 'cuda', 'auto'], default='cpu')
    args = parser.parse_args()
    report = run(args.traces, args.out, args.seeds, args.epochs, args.device)
    print(json.dumps(report['aggregate'], indent=2))


if __name__ == '__main__':
    main()

```

## Limitations and next steps
- This is an adaptation of CSRT tracking, not a full JoulesEye reproduction.
  No synchronized RGB, automatic initialization, or detector-based recovery is available.
- Fixed manual initial boxes, possible undetected drift and box-scale changes
  need annotation-based evaluation. The sampled overlays are qualitative only.
- Four participants with one session each cannot establish generalization.
  Repeated seeds assess training variability, not population uncertainty.
- Marker semantics, row synchronization and the assumed 60-second motion boundary
  require verification. `anestis` is still excluded.
- Compare error together with coverage; any abstentions must remain visible.
- Next: annotate nose locations and drift during motion, verify references,
  test recovery without label-based tuning, and expand the independent test set.
  Keep the median and spectral baselines and retain both crop variants for review.